In [ ]:
import json
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import *

# Read from Bronze
df_bronze = spark.read.format("delta").load("abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Bronze_LH.Lakehouse/Tables/bronze_line_status")

# Define improved schema
schema = StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("modeName", StringType(), True),
    StructField("lineStatuses", ArrayType(StructType([
        StructField("statusSeverity", IntegerType(), True),
        StructField("statusSeverityDescription", StringType(), True),
        StructField("reason", StringType(), True),
        StructField("validityPeriods", ArrayType(StructType([
            StructField("fromDate", StringType(), True),
            StructField("toDate", StringType(), True),
            StructField("isNow", BooleanType(), True)
        ])), True),
        StructField("disruption", StructType([
            StructField("category", StringType(), True),
            StructField("closureText", StringType(), True)
        ]), True)
    ])), True)
])

# Parse JSON and extract all useful fields
df_silver = df_bronze \
    .withColumn("parsed", from_json(col("raw_json"), schema)) \
    .select(
        col("parsed.id").alias("line_id"),
        col("parsed.name").alias("line_name"),
        col("parsed.modeName").alias("mode_name"),
        col("parsed.lineStatuses")[0]["statusSeverity"].alias("status_severity"),
        col("parsed.lineStatuses")[0]["statusSeverityDescription"].alias("status_description"),
        col("parsed.lineStatuses")[0]["reason"].alias("disruption_reason"),
        col("parsed.lineStatuses")[0]["validityPeriods"][0]["fromDate"].alias("disruption_from"),
        col("parsed.lineStatuses")[0]["validityPeriods"][0]["toDate"].alias("disruption_to"),
        col("parsed.lineStatuses")[0]["validityPeriods"][0]["isNow"].alias("is_active_disruption"),
        col("parsed.lineStatuses")[0]["disruption"]["category"].alias("disruption_category"),
        col("parsed.lineStatuses")[0]["disruption"]["closureText"].alias("closure_text"),
        col("ingestion_timestamp")
    )

# Remove nulls and duplicates
df_silver = df_silver \
    .dropna(subset=["line_id", "line_name"]) \
    .dropDuplicates(["line_id", "ingestion_timestamp"])

# Save to Silver Lakehouse
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Silver_LH.Lakehouse/Tables/silver_line_status")
print(f"✅ {df_silver.count()} records written to Silver!")

StatementMeta(, 09f6700e-e32d-45c9-a723-3ed3756c3016, 5, Finished, Available, Finished, False)

✅ 836 records written to Silver!


In [4]:
# Preview improved Silver data
df_check = spark.read.format("delta").load("abfss://TfL-Analytics@onelake.dfs.fabric.microsoft.com/TfL_Silver_LH.Lakehouse/Tables/silver_line_status")

df_check.filter(col("disruption_reason").isNotNull()).show(3, truncate=False)

StatementMeta(, 09f6700e-e32d-45c9-a723-3ed3756c3016, 6, Finished, Available, Finished, False)

+--------+---------+---------+---------------+------------------+-----------------------------------------------------------+--------------------+--------------------+--------------------+-------------------+-------------+--------------------------+
|line_id |line_name|mode_name|status_severity|status_description|disruption_reason                                          |disruption_from     |disruption_to       |is_active_disruption|disruption_category|closure_text |ingestion_timestamp       |
+--------+---------+---------+---------------+------------------+-----------------------------------------------------------+--------------------+--------------------+--------------------+-------------------+-------------+--------------------------+
|bakerloo|Bakerloo |tube     |20             |Service Closed    |Bakerloo Line: Service will resume later this morning.\r\n |2026-06-11T00:14:33Z|2026-06-11T03:20:23Z|true                |RealTime           |serviceClosed|2026-06-11 00:21:15.852417|
